# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-01a_build_from_scratch.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker (or [03_own_environment.md](https://github.com/justkrismanohar/fine-tuning-build/blob/main/03_own_environment.md)).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day1/T1-01a_build_from_scratch.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [ ]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day1"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day1" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)


On this track you **walk the hosting steps** (check → weights on disk → API door) with simulated output. You do not download models. Circle ‘demo replies’ on the write-up, not ‘I started vLLM’.


# T1-01a Build OLMo and Phi from scratch

**Time:** 09:15–09:50 (first half of the T1-01 block) &nbsp;|&nbsp; **Evidence:** `logs/endpoint.json` and a passing health check

**Same steps as the script:** `python build_llms.py check|download|serve|health|stop`

**Notebook learning outcomes**

- Say what “from scratch” means here: **deploy and serve**, not pre-train a foundation model.
- Check the host (GPU, Docker, Python packages).
- Fetch (or confirm) OLMo and Phi **weights on disk**.
- Start an OpenAI-compatible API for each model.
- Record the URLs so T1-01 can call them.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers — the big picture

A language model is three boxes:

1. **Training data** — the pile of text (you inspect this as a licence question in T1-04).
2. **Weights** — that pile compressed into files on disk. OLMo and Phi are two different files.
3. **Output** — produced at use time, when a **runtime** loads the weights and an **API** accepts a prompt.

This notebook is how box 2 gets onto a computer you control, and how a door (the API) is opened in front of it. That is **N7**: if the work is core (CPO, TTDF, Finance, THA, anyone with PII), you should be able to start this door yourself rather than paste into ChatGPT.

You are **not** training OLMo. The Instruct weights already exist. You download them, load them, serve them.

If this machine has no GPU, you still run the **check** cell, then skip serve, and T1-01 will use `BACKEND='demo'` or the shared course URL. Managers: running check and reading the health output is the point. Deep-code people may prefer the script in a terminal; it is the same functions.


## Ministry link

| EOI type | Why this notebook exists |
|---|---|
| **CPO / TTDF / Finance / THA** | You asked for on-prem or Government-controlled inference. Starting vLLM (or the course image) is that skill, at classroom size. |
| **Trade IMSD** | The “agent” is this API plus rules. Today you stand the API up. Tools (GLPI, password reset) stay off. |
| **Labour / Health / OPR** | The model you will later ask to *quote a criterion* has to live somewhere. Today you see where. |

Equivalent script, if you would rather not click cells:

```text
python build_llms.py check
python build_llms.py download
python build_llms.py serve --engine vllm
python build_llms.py wait
```

On Windows without NVIDIA/WSL, `check` will tell you to stay on demo or use Docker/WSL2. That is an honest result, not a fail.


## 1. Look at the host

Before you buy or start anything, ask: is there a GPU, what is it, and will a Linux runtime (vLLM) actually run? T1-04 expands the GPU questions. Here you only collect facts.


In [ ]:
# [RUN] COLAB TRACK — simulate check(); we do not start a GPU runtime
import json
import civicdesk_lab as lab

info = lab.simulate_hosting_check()


### What you should have seen

- `nvidia_smi`: a name, VRAM, compute capability — or null. Null means this laptop will not host the classroom vLLM.
- `vllm_import`: True only on a Linux+CUDA Python where vLLM is installed (course image).
- `docker`: True if you can run the official vLLM container instead.
- `transformers_import`: a slower fallback (`--engine hf`).

**Link:** standing inference (this chat API) is what we start today. A later LoRA job is a **burst** on a possibly bigger card, then that card is stopped. Do not buy one box “for AI” without that split (T1-04).


## 2. Choose an engine

`# [CHANGE]` one value. Everyone else can leave `skip` and go to T1-01 on demo/shared.

| ENGINE | What it does | When |
|---|---|---|
| `vllm` | Starts two OpenAI servers (OLMo :8000, Phi :8001) | Course image / Linux+CUDA with `pip install vllm` |
| `docker` | Same, via `vllm/vllm-openai` containers | GPU + Docker, no local pip |
| `hf` | Tiny transformers server (slow on CPU) | Review / no vLLM |
| `skip` | Do not start anything | No GPU; use demo or shared in T1-01 |


In [ ]:
# COLAB TRACK: engine is locked. Docker course may use vllm / docker / hf.
ENGINE = "skip"
MODELS = ["olmo", "phi"]
print("ENGINE =", ENGINE, "(locked on Colab track)")
print("You still learn the hosting picture. You do not fetch weights here.")


## 3. Weights on disk

Downloading is not training. It copies the published files (Apache-2.0 OLMo, MIT Phi as published — confirm the live card in T1-04) into the Hugging Face cache. The course image may already have them. If `ENGINE='skip'`, do not wait on this cell.


In [ ]:
# [RUN] COLAB TRACK — do not download OLMo/Phi (several GB)
print("skipped download")
print("Official ids (Docker course fetches these):")
print("  allenai/OLMo-2-0425-1B-Instruct")
print("  microsoft/Phi-4-mini-instruct")


### After download

You now have **box 2** locally. Until a runtime loads those files, nothing answers. ChatGPT never gives you this folder. That is the open-weight difference (T1-04).


## 4. Start the door (the API)

Each model gets its own process and port. vLLM speaks the same `/v1/chat/completions` contract OpenAI uses, so T1-01 does not care which engine you picked.

This cell starts servers **in the background**. Leave them running. Stop later with `build.stop()` or `python build_llms.py stop`.


In [ ]:
# [RUN] COLAB TRACK — no :8000 / :8001
import civicdesk_lab as lab

print("No local server. T1-01 will use canned [demo] replies.")
lab.simulate_hosting_skip_serve()


### What “healthy” means

`GET http://127.0.0.1:8000/v1/models` lists OLMo. `:8001` lists Phi. T1-01 will POST chat completions to those URLs.

If wait timed out: open `logs/olmo_*.log`, or use demo. A fluent demo reply is **not** evidence of a live model — look for `"mode": "live"` in T1-01.

**Trade / Labour:** the door is up. The model still must not reset a password or hire anyone. Rules live in the prompt (today) and in tool allowlists (Day 3), not in the GPU.


In [ ]:
# [RUN] write / confirm the file T1-01 will read
from pathlib import Path
p = Path("logs/endpoint.json")
print(p.read_text(encoding="utf-8") if p.exists() else "no endpoint.json (ok if ENGINE=skip)")


In [ ]:
# [OPTIONAL] skipped — there is no live /v1 on this track
print("olmo", "demo (no HTTP)")
print("phi ", "demo (no HTTP)")


## Write-up (everyone, including skip)

1. In one line: data → **weights** → runtime → API → output. Circle which of those you personally started today.
2. Why two ports, not one process? (Hint: this course compares OLMo and Phi on the same prompt after lunch.)
3. If you skipped: which backend will you use in T1-01, and who owns that computer?

Go on to **T1-01_first_call.ipynb**. Keep the servers running.
